# 结果后处理

**常见用法**：截断超长输出（日志/网页正文/大 JSON，防上下文爆炸）、
敏感字段脱敏（手机号/邮箱/密钥打码）、统一格式化（时间/编码/多余空白清理）。

**钩子内的做法**：
- `res = execute(request)` 拿到结果后检查/修改 content，**重建 ToolMessage** 返回
  （消息对象不可原地改；重建时 name/tool_call_id/status 原样带上，协议绑定不丢）
- 截断文案要给模型指路（如"已截断，请缩小查询范围"），模型会主动引导用户而不是瞎编
- 脱敏用正则替换 content 即可，套路与截断相同

In [2]:
from typing import Literal

from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage, ToolMessage
from langchain_core.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, MessagesState, StateGraph
from langgraph.prebuilt import ToolNode
from langgraph.prebuilt.tool_node import ToolCallRequest
from rich import print

load_dotenv(override=True)

model = init_chat_model(
    model_provider="deepseek",
    model="deepseek-flash",
    extra_body={"thinking": {"type": "disabled"}}
)

MAX_CHARS = 300


@tool
def read_app_log(keyword: str) -> str:
    """读取应用日志中含关键词的内容"""
    line = f"[2026-09-18 10:23:01] ERROR order-service: 未找到含 '{keyword}' 的记录，重试 3 次失败\n"
    return line * 40  # 模拟 40 行约 4600 字的日志


def truncate_result(request: ToolCallRequest, execute) -> ToolMessage | object:
    """结果后处理：超长输出截断，保留 tool_call_id 绑定"""
    res = execute(request)
    if len(res.content) > MAX_CHARS:
        print(f"[截断] {res.content[:20]}... 共{len(res.content)}字 -> 保留前{MAX_CHARS}字")
        return ToolMessage(
            content=(res.content[:MAX_CHARS] + f"\n...(日志共 {len(res.content)} 字符已截断，需要更多请缩小查询范围)"),
            name=res.name,
            tool_call_id=res.tool_call_id,
            status=res.status
        )
    return res


model_with_tools = model.bind_tools([read_app_log])
tool_node = ToolNode([read_app_log], wrap_tool_call=truncate_result)


class ChatState(MessagesState):
    pass


def llm_node(state: ChatState) -> dict:
    return {"messages": [model_with_tools.invoke(state["messages"])]}


def router(state: ChatState) -> Literal["tool_node", END]:
    return "tool_node" if state["messages"][-1].tool_calls else END


builder = StateGraph(state_schema=ChatState)
builder.add_node("llm_node", llm_node)
builder.add_node("tool_node", tool_node)
builder.add_edge(START, "llm_node")
builder.add_conditional_edges("llm_node", router, [END, "tool_node"])
builder.add_edge("tool_node", "llm_node")
graph = builder.compile(checkpointer=InMemorySaver())

res = graph.invoke(
    {
        "messages": [HumanMessage("帮我看看日志里关于 '订单超时' 的情况")]
    },
    config={"configurable": {"thread_id": "1"}}
)

[截断] [2026-09-18 10:23:01... 共2720字 -> 保留前300字

In [3]:
print(res)

{
    'messages': [
        HumanMessage(
            content="帮我看看日志里关于 '订单超时' 的情况",
            additional_kwargs={},
            response_metadata={},
            id='c0b0811f-e324-47d5-8242-b1f190660601'
        ),
        AIMessage(
            content="I'll search the application logs for entries related to '订单超时' (order timeout).",
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 58,
                    'prompt_tokens': 277,
                    'total_tokens': 335,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {
                        'audio_tokens': None,
                        'cache_write_tokens': None,
                        'cached_tokens': 128,
                        'image_tokens': None,
                        'text_tokens': None
                    },
                    'prompt_cache_hit_tokens': 128,
                    'prompt_cache_miss_tokens': 149
                },
                'model_provider': 'deepseek',
                'model_name': 'deepseek-flash',
                'system_fingerprint': 'aeb56401ca74e127821c4f9126dcb669',
                'id': 'fd437427-a4f8-4622-9a88-bf35f2236fae',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a0b0b2-958c-7523-8944-ed2582b057fb-0',
            tool_calls=[
                {
                    'name': 'read_app_log',
                    'args': {'keyword': '订单超时'},
                    'id': 'call_00_2Dedq2STVkSx4vGvmTLr4113',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 277,
                'output_tokens': 58,
                'total_tokens': 335,
                'input_token_details': {'cache_read': 128},
                'output_token_details': {}
            }
        ),
        ToolMessage(
            content="[2026-09-18 10:23:01] ERROR order-service: 未找到含 '订单超时' 的记录，重试 3 
次失败\n[2026-09-18 10:23:01] ERROR order-service: 未找到含 '订单超时' 的记录，重试 3 次失败\n[2026-09-18 10:23:01]
ERROR order-service: 未找到含 '订单超时' 的记录，重试 3 次失败\n[2026-09-18 10:23:01] ERROR order-service: 未找到含
'订单超时' 的记录，重试 3 次失败\n[2026-09-18 10:23:01] ERROR \n...(日志共 2720 
字符已截断，需要更多请缩小查询范围)",
            name='read_app_log',
            id='01d2ba8f-19d3-4e76-acd3-40179b0bb401',
            tool_call_id='call_00_2Dedq2STVkSx4vGvmTLr4113'
        ),
        AIMessage(
            content='我用**中文**关键词 `订单超时` 查了一次，结果并不理想，情况如下：\n\n## 
查询结果\n\n日志系统返回的是**"未找到含 \'订单超时\' 的记录，重试 3 
次失败"**，也就是说——**没有查到任何真正关于"订单超时"的日志内容**。\n\n返回的这几条恰恰是"查询失败"的记录本身（orde
r-service 连续报了 4 次重试失败），而不是业务日志。\n\n## 需要说明的几点\n\n1. 
**关键词可能不匹配**：日志里实际用的措辞可能不是中文"订单超时"，而是英文或其它写法，比如：\n   - `order timeout` / 
`order_timeout`\n   - `timeout` / `timed out`\n   - `ORDER_TIMEOUT`\n   - `超时`（去掉"订单"两字）\n\n2. 
**可能是日志本身有问题**：连续 4 次"重试 3 
次失败"通常意味着日志服务/采集链路异常（索引不可用、写入延迟、服务重启等），而不是真的没有这条业务日志。\n\n3. 
**结果被截断了**：系统提示"日志共 2720 字符已截断"，说明底层确实有数据，只是命中方式和展示受限。\n\n## 
建议下一步\n\n我可以帮你做以下任意一项，你告诉我选哪个（或直接给更精确的词）：\n\n- 用 `timeout` 再查一次\n- 用 
`超时` 单独查\n- 用 `order` 查，看看订单服务整体的报错情况\n- 用 `重试失败` 
查，先定位日志链路是不是真的坏了\n\n你要我先试哪个？我个人建议先查 **`timeout`**，同时查 **`重试失败`** 
确认日志系统是否正常。',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 385,
                    'prompt_tokens': 532,
                    'total_tokens': 917,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {
                        'audio_tokens': None,
                        'cache_write_tokens': None,
                        'cached_tokens': 256,
                        'image_tokens': None,
                        'text_tokens': None
                    },
                    'prompt_cache_hit_tokens': 256,
  